# 7.5 Text Mining in Higher Ed – Sentiment Analysis — Code Brief

## Key Concepts

- **Sentiment analysis** estimates emotional tone (positive/negative/neutral) of text.
- **VADER** — lexicon-based, rule-driven, no training data needed. Fast baseline. Compound score: ≥0.05 positive, ≤-0.05 negative, else neutral.
- Supervised alternative: TF-IDF + Logistic Regression, trained on labeled data.
- Validation pattern: crosstab VADER labels against ground-truth labels — diagonal = agreement, off-diagonal = where VADER misses nuance.
- Ethics reminder: sentiment scores are a signal for further investigation, not a definitive judgment.

## Setup and Data Preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import random
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pd.options.display.max_columns = None


In [ ]:
filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
ML_Survey_Data = pd.read_csv(f'{filepath}ML_Survey_Data.csv')
display(ML_Survey_Data)

## Lexicon-Based Sentiment (VADER)

In [ ]:
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

df_Sentiment = ML_Survey_Data[['SID','Free_Response_Text']]

df_Sentiment['vader_compound'] = df_Sentiment['Free_Response_Text'].apply(lambda t: sia.polarity_scores(t)['compound'])

# Convert compound score to a simple label
def vader_label(score, pos_thresh=0.05, neg_thresh=-0.05):
    if score >= pos_thresh:
        return 'positive'
    if score <= neg_thresh:
        return 'negative'
    return 'neutral'

df_Sentiment['vader_label'] = df_Sentiment['vader_compound'].apply(vader_label)
display(df_Sentiment[['Free_Response_Text','vader_compound','vader_label']].head(10))


## Interpreting and Communicating Results

In [ ]:
sent_counts = df_Sentiment['vader_label'].value_counts().reset_index()
sent_counts.columns = ['vader_label','Count']
fig = px.bar(sent_counts, x='vader_label', y='Count', title='Sentiment Distribution (VADER)')
fig.show()


## Custom Sentiment & Validation

In [ ]:
# Synthetic sentiment-linked comments
df_training = pd.read_csv(f'{filepath}training.csv')

sent_df = df_training[['SID']].copy()

positive_bank = [
    'I really enjoyed this course and felt supported by my instructors.',
    'The assignments were challenging in a good way, and I learned a lot.',
    'Advising was helpful and I feel confident about my academic plan.',
    'I feel connected on campus and my classes have been engaging.'
]
neutral_bank = [
    'The semester was okay overall, with some ups and downs.',
    'Some parts of the course were useful, and others were less clear.',
    'I managed my workload, but it was sometimes difficult to stay organized.',
    'My experience was mixed depending on the class and the week.'
]
negative_bank = [
    'I struggled a lot and often felt like I did not know where to get help.',
    'The workload was overwhelming and I felt stressed most weeks.',
    'I had a hard time understanding expectations and felt unsupported.',
    'I felt disconnected and frustrated with how things were communicated.'
]

def make_sentiment_comment(gpa_norm):
    # More likely positive when GPA_norm is higher; more likely negative when lower
    if gpa_norm > 3.7:
        label = 'positive'
        text = random.choice(positive_bank)
    elif gpa_norm < 3.4:
        label = 'negative'
        text = random.choice(negative_bank)
    else:
        # mixed middle group
        label = random.choice(['neutral','positive','negative'])
        text = random.choice(neutral_bank if label=='neutral' else (positive_bank if label=='positive' else negative_bank))
    return text, label

rows=[]
for sid, g in zip(df_training['SID'], df_training['HS_GPA']):
    text, label = make_sentiment_comment(float(g))
    rows.append((sid, text, label))

sent_df = pd.DataFrame(rows, columns=['SID','Comment','Labeled Sentiment'])

sent_df1 = pd.merge(sent_df,df_Sentiment[['SID', 'Free_Response_Text', 'vader_label']],on="SID",how="left")
display(sent_df1.head())
sent_df1['Labeled Sentiment'].value_counts()

## Model Comparison (Crosstab Analysis)

In [ ]:
ct = pd.crosstab(sent_df1['Labeled Sentiment'], sent_df1['vader_label'])
ct

In [ ]:
pd.set_option('display.max_colwidth', None)
sent_df1